# LightMamba-ASL — Google Colab Training
### GPU-accelerated training with native mamba_ssm
**Steps:** Mount Drive → Clone Repo → Link Dataset → Install → Train → Download Checkpoint

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive (your videos folder must be here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Clone your GitHub repo
!git clone https://github.com/Niranjanprakash/ASL_Sign_Lightmamba.git
%cd ASL_Sign_Lightmamba

In [ ]:
# Step 4: Link dataset from Drive to project
# CHANGE THIS PATH to where you uploaded videos in Google Drive
DRIVE_VIDEOS_PATH = '/content/drive/MyDrive/WLASL/videos'  # <-- change this
DRIVE_JSON_PATH   = '/content/drive/MyDrive/WLASL/WLASL_v0.3.json'  # <-- change this

import os
os.makedirs('dataset/raw/videos', exist_ok=True)
os.makedirs('dataset/metadata', exist_ok=True)

# Symlink videos folder (faster than copying)
if not os.path.exists('dataset/raw/videos_linked'):
    !ln -s {DRIVE_VIDEOS_PATH} dataset/raw/videos_linked
    print('Videos linked!')

# Copy JSON metadata
!cp {DRIVE_JSON_PATH} dataset/metadata/WLASL_v0.3.json
print('Metadata copied!')

# Point VIDEO_DIR to symlinked folder
# Update config to use linked path
!sed -i 's|VIDEO_DIR = DATASET_ROOT / "raw" / "videos"|VIDEO_DIR = DATASET_ROOT / "raw" / "videos_linked"|' backend/config.py
print('Config updated!')

In [ ]:
# Step 5: Install dependencies
!pip install -r requirements.txt -q

# Install native mamba_ssm (works on Colab GPU!)
!pip install mamba-ssm -q
print('All dependencies installed!')

In [ ]:
# Step 6: Verify GPU + mamba_ssm
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

try:
    import mamba_ssm
    print('mamba_ssm: NATIVE (fast!)')
except:
    print('mamba_ssm: Fallback (PyTorch)')

In [ ]:
# Step 7: Prepare dataset (splits + landmark cache)
!python -m backend.data.prepare_dataset

In [ ]:
# Step 8: Train the model
!python -m backend.training.train

In [ ]:
# Step 9: Evaluate on test set
!python -m backend.evaluation.evaluate

In [ ]:
# Step 10: Save checkpoint to Google Drive
import shutil
SAVE_PATH = '/content/drive/MyDrive/WLASL/best_model_104classes.pth'
shutil.copy('checkpoints/best_model.pth', SAVE_PATH)
print(f'Checkpoint saved to Drive: {SAVE_PATH}')

In [ ]:
# Step 11: Download checkpoint directly to your PC
from google.colab import files
files.download('checkpoints/best_model.pth')